Inference is the process of loading a previously trained Machine Learning (ML) model and running it to calculate a prediction on new, unlabeled input data. It's the step where the model transitions from a research artifact to a piece of production software.

In [ ]:
#install required libraries
!pip install pandas pandas-datareader yfinance pandas-ta-classic numpy torch pytorch-forecasting python-dateutil


In [6]:
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import pandas_ta_classic as ta
import numpy as np
import torch
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from datetime import datetime
from dateutil.relativedelta import relativedelta

# --- 1. CONFIGURATION ---
MODEL_PATH = "tft_model.ckpt"
CURRENCY_PAIR = 'EURUSD=X'
FRED_CODES = {
    'FEDFUNDS': 'US_Interest_Rate',
    'CPIAUCSL': 'US_Inflation',
    'UNRATE': 'US_Unemployment',
    'PAYEMS': 'US_NonFarm_Payrolls',
    'GDPC1': 'US_GDP',
    'BOPGSTB': 'US_Trade_Balance',
    'VIXCLS': 'Global_VIX',
    'DCOILWTICO': 'WTI_Crude_Oil_Price'
}
TARGET_COL = 'EURUSD=X' # Note: In the logic below, we rename this to Close_Price
FREQ = 'M'

class ForexForecaster:
    def __init__(self, model_path):
        print(f"Loading model from {model_path}...")
        # Load on CPU
        self.model = TemporalFusionTransformer.load_from_checkpoint(model_path, map_location=torch.device('cpu'))
        self.model.eval()
        
    def fetch_recent_data(self, lookback_months=60):
        """
        Fetches historical data and ensures strict column naming.
        """
        end_date = datetime.today()
        start_date = end_date - relativedelta(months=lookback_months)
        
        print(f"Fetching live data from {start_date.date()} to {end_date.date()}...")

        # 1. Fetch Forex (Robust Method)
        # auto_adjust=True removes the warning and handles splits/dividends (though rare for Forex)
        raw_yfinance = yf.download(CURRENCY_PAIR, start=start_date, end=end_date, interval='1d', progress=False, auto_adjust=True)
        
        # Handle yfinance MultiIndex columns (The root cause of your error)
        # Check if columns have levels (Price Type, Ticker)
        if isinstance(raw_yfinance.columns, pd.MultiIndex):
            try:
                # Try to extract 'Close'
                df_fx = raw_yfinance.xs('Close', level=0, axis=1)
            except KeyError:
                # Fallback if 'Close' isn't top level
                df_fx = raw_yfinance['Close']
        else:
            df_fx = raw_yfinance[['Close']]

        # Force valid DataFrame structure
        if isinstance(df_fx, pd.Series):
            df_fx = df_fx.to_frame()
            
        # If the column is still named strictly after the ticker, rename it
        # We assume the first column is the Close price
        df_fx = df_fx.iloc[:, [0]] 
        df_fx.columns = ["Close_Price"]
        
        # Remove Timezone info to match FRED data (FRED is usually naive)
        df_fx.index = df_fx.index.tz_localize(None)

        # 2. Fetch Macro (FRED)
        try:
            df_fred = web.DataReader(list(FRED_CODES.keys()), 'fred', start_date, end_date)
            df_fred.columns = list(FRED_CODES.values())
        except Exception as e:
            print(f"Warning: FRED fetch failed ({e}). Using zeros.")
            df_fred = pd.DataFrame(columns=FRED_CODES.values())

        # 3. Combine
        df = df_fx.join(df_fred, how='outer')
        
        # 4. Fill NaNs
        df = df.ffill().bfill()
        
        # Debug print to ensure column exists
        if 'Close_Price' not in df.columns:
            print("CRITICAL ERROR: 'Close_Price' column missing after join.")
            print(f"Columns found: {df.columns}")
        
        return df

    def preprocess_for_inference(self, df):
        df = df.copy()

        # Check for column existence before proceeding
        if 'Close_Price' not in df.columns:
            raise KeyError(f"'Close_Price' not found. Available columns: {df.columns.tolist()}")

        # --- A. Technical Indicators ---
        df['Lag_1D'] = df['Close_Price'].shift(1)
        df['SMA_20'] = ta.sma(df['Close_Price'], length=20)
        df['SMA_50'] = ta.sma(df['Close_Price'], length=50)
        df['RSI_14'] = ta.rsi(df['Close_Price'], length=14)
        df['Historical_Volatility_20D'] = df['Close_Price'].pct_change().rolling(window=20).std() * (252**0.5)

        # --- B. Resample to Monthly ---
        # Initialize resampled dataframe
        df_res = pd.DataFrame()
        
        # Resample Close_Price specifically
        df_res['EURUSD=X'] = df['Close_Price'].resample(FREQ).last()
        
        # Resample other columns
        # Note: We skip 'Close_Price' in the loop because we renamed it to 'EURUSD=X' above 
        # to match the Target name expected by the model during training.
        skip_cols = ['Close_Price']
        
        for col in df.columns:
            if col not in skip_cols:
                df_res[col] = df[col].resample(FREQ).last()
        
        df_res = df_res.ffill().dropna()

        # --- C. Feature Engineering (Lags) ---
        numeric_cols = df_res.select_dtypes(include=np.number).columns
        for col in numeric_cols:
            df_res[f'{col}_Lag1'] = df_res[col].shift(1)
            df_res[f'{col}_Lag4'] = df_res[col].shift(4)

        # --- D. Stationarity (Differencing) ---
        # Capture last state for reconstruction
        last_actual_price = df_res['EURUSD=X'].iloc[-1]
        last_actual_date = df_res.index[-1]

        # Apply differencing to non-lagged columns
        cols_to_diff = [c for c in df_res.columns if 'Lag' not in c] 
        
        for col in cols_to_diff:
            df_res[f'{col}_D1'] = df_res[col].diff()

        # Drop NaNs created by lags/diffs
        df_res = df_res.dropna()

        return df_res, last_actual_price, last_actual_date

    def predict_next_month(self):
        # 1. Get Data
        raw_df = self.fetch_recent_data()
        
        # 2. Process
        processed_df, last_price, last_date = self.preprocess_for_inference(raw_df)
        
        print(f"\nContext: Last closed month data available: {last_date.date()}")
        print(f"Context: Last Price: {last_price:.4f}")

        # 3. Prepare for PyTorch Forecasting
        # Create Dummy Row for next month to provide a target container
        new_row = processed_df.iloc[[-1]].copy()
        new_row.index = [last_date + relativedelta(months=1)]
        
        inference_df = pd.concat([processed_df, new_row])
        
        # Add required TFT columns
        inference_df['time_idx'] = np.arange(len(inference_df))
        inference_df['series'] = 'EURUSD'
        inference_df = inference_df.reset_index().rename(columns={'index': 'date'})

        # 4. Create Prediction Dataset
        try:
            # --- FIX IS HERE ---
            # Use 'from_parameters' instead of 'from_dataset' because dataset_parameters is a dict
            pred_ds = TimeSeriesDataSet.from_parameters(
                self.model.dataset_parameters, 
                inference_df, 
                predict=True, 
                stop_randomization=True
            )
            
            # Create dataloader
            pred_loader = pred_ds.to_dataloader(batch_size=1, shuffle=False)
            
            # 5. Predict
            # return_x=False returns only the predictions
            raw_prediction = self.model.predict(pred_loader, mode="prediction", return_x=False)
            
            # Extract the actual value (Batch 0, Step 0)
            predicted_change = raw_prediction[0][0].item() 
            
            # 6. Reconstruct Price
            predicted_price = last_price + predicted_change
            
            return {
                "target_date": (last_date + relativedelta(months=1)).date(),
                "last_price": last_price,
                "predicted_change": predicted_change,
                "predicted_price": predicted_price
            }

        except Exception as e:
            print(f"Inference Error: {e}")
            import traceback
            traceback.print_exc()
            return None
# --- EXECUTION ---
# if __name__ == "__main__":
#     print("--- starting inference pipeline ---")
#     forecaster = ForexForecaster(MODEL_PATH)
#     result = forecaster.predict_next_month()
    
#     if result:
#         print("\n" + "="*40)
#         print(f"FORECAST REPORT FOR: {result['target_date'].strftime('%B %Y')}")
#         print("="*40)
#         print(f"Current Price:          {result['last_price']:.4f}")
#         print(f"Predicted Change:       {result['predicted_change']:.4f}")
#         print("-" * 40)
#         print(f"FINAL PREDICTION:       {result['predicted_price']:.4f}")
#         print("="*40)

C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



--- starting inference pipeline ---
Loading model from tft_model.ckpt...


C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


Fetching live data from 2020-12-01 to 2025-12-01...


C:\Users\Vipin\AppData\Local\Temp\ipykernel_64652\3689960390.py:111: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

C:\Users\Vipin\AppData\Local\Temp\ipykernel_64652\3689960390.py:120: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

C:\Users\Vipin\AppData\Local\Temp\ipykernel_64652\3689960390.py:120: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

C:\Users\Vipin\AppData\Local\Temp\ipykernel_64652\3689960390.py:120: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

C:\Users\Vipin\AppData\Local\Temp\ipykernel_64652\3689960390.py:120: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

C:\Users\Vipin\AppData\Local\Temp\ipykernel_64652\3689960390.py:120: FutureWarning:

'M' is deprecated and will be removed in a future version, please


Context: Last closed month data available: 2025-11-30
Context: Last Price: 1.1600


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.



FORECAST REPORT FOR: December 2025
Current Price:          1.1600
Predicted Change:       -0.0033
----------------------------------------
FINAL PREDICTION:       1.1567


In [1]:
# -*- coding: utf-8 -*-
"""
Created on Sat Nov 29 22:39:26 2025

@author: sweth
"""

import dash
from dash import html, Output, Input

# Initialize app
app = dash.Dash(__name__)
app.title = "Predicting Currency Movements"
forecaster = ForexForecaster(MODEL_PATH)

def predict_currency():
    return forecaster.predict_next_month()['predicted_price'] #we are using class from above to predict, MAKE SURE TO RUN PREVIOUS CELL

# Layout
app.layout = html.Div(
    style={
        'backgroundColor': '#2C3E50',
        'padding': '30px',
        'borderRadius': '12px',
        'textAlign': 'center',
        'maxWidth': '900px',
        'margin': '30px auto'
    },
    children=[
        html.Div(
            style={'display': 'flex', 'alignItems': 'center',
                   'justifyContent': 'center', 'gap': '15px'},
            children=[
                html.Img(
                    src='https://img.icons8.com/ios-filled/100/ffffff/currency-exchange.png',
                    style={'height': '60px'}
                ),
                html.H1(
                    "Predicting Currency Movements",
                    style={'color': '#FFFFFF', 'fontWeight': '700',
                           'fontSize': '2.8rem', 'margin': '0'}
                )
            ]
        ),

        # ---------- BUTTON ----------
        html.Button(
            "Run Prediction",
            id="predict-btn",
            n_clicks=0,
            style={
                'marginTop': '30px',
                'padding': '12px 25px',
                'fontSize': '1.2rem',
                'borderRadius': '8px',
                'cursor': 'pointer'
            }
        ),

        # ---------- OUTPUT AREA ----------
        html.Div(
            id="prediction-output",
            style={'color': 'white', 'marginTop': '25px', 'fontSize': '1.4rem'}
        )
    ]
)

# ------------- Callback (button → function → output) -------------
@app.callback(
    Output('prediction-output', 'children'),
    Input('predict-btn', 'n_clicks')
)
def update_prediction(n):
    if n > 0:
        return predict_currency()   
    return ""


# Run the app
if __name__ == '__main__':
    app.run(debug=True)


NameError: name 'ForexForecaster' is not defined

In [3]:
import dash
from dash import dcc, html, Input, Output, State, callback_context
import plotly.graph_objects as go
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import pandas_ta_classic as ta
import numpy as np
import torch
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import warnings

# Suppress warnings for cleaner logs
warnings.filterwarnings("ignore")

# --- 1. CONFIGURATION ---
MODEL_PATH = "tft_model.ckpt"
CURRENCY_PAIR_SOURCE = 'EURUSD=X'
FRED_CODES = {
    'FEDFUNDS': 'US_Interest_Rate',
    'CPIAUCSL': 'US_Inflation',
    'UNRATE': 'US_Unemployment',
    'PAYEMS': 'US_NonFarm_Payrolls',
    'GDPC1': 'US_GDP',
    'BOPGSTB': 'US_Trade_Balance',
    'VIXCLS': 'Global_VIX',
    'DCOILWTICO': 'WTI_Crude_Oil_Price'
}
FREQ = 'M'

# --- 2. BACKEND LOGIC ---
class ForexForecaster:
    def __init__(self, model_path):
        self.model_loaded = False
        try:
            print(f"Loading model from {model_path}...")
            # Load on CPU
            self.model = TemporalFusionTransformer.load_from_checkpoint(model_path, map_location=torch.device('cpu'))
            self.model.eval()
            self.model_loaded = True
        except Exception as e:
            print(f"Warning: Model not found or failed to load ({e}). Dashboard will run in Demo Mode.")
            self.model = None

    def fetch_recent_data(self, lookback_months=60):
        end_date = datetime.today()
        start_date = end_date - relativedelta(months=lookback_months)
        
        # 1. Fetch Forex
        raw_yfinance = yf.download(CURRENCY_PAIR_SOURCE, start=start_date, end=end_date, interval='1d', progress=False, auto_adjust=True)
        
        # Handle MultiIndex
        if isinstance(raw_yfinance.columns, pd.MultiIndex):
            try:
                df_fx = raw_yfinance.xs('Close', level=0, axis=1)
            except KeyError:
                df_fx = raw_yfinance['Close']
        else:
            df_fx = raw_yfinance[['Close']]

        if isinstance(df_fx, pd.Series):
            df_fx = df_fx.to_frame()
            
        df_fx = df_fx.iloc[:, [0]] 
        df_fx.columns = ["Close_Price"]
        df_fx.index = df_fx.index.tz_localize(None)

        # 2. Fetch Macro (FRED)
        try:
            df_fred = web.DataReader(list(FRED_CODES.keys()), 'fred', start_date, end_date)
            df_fred.columns = list(FRED_CODES.values())
        except Exception:
            df_fred = pd.DataFrame(columns=FRED_CODES.values())

        # 3. Combine & Fill
        df = df_fx.join(df_fred, how='outer')
        df = df.ffill().bfill()
        
        return df

    def preprocess_for_inference(self, df):
        df = df.copy()
        
        # Indicators
        df['Lag_1D'] = df['Close_Price'].shift(1)
        df['SMA_20'] = ta.sma(df['Close_Price'], length=20)
        df['SMA_50'] = ta.sma(df['Close_Price'], length=50)
        df['RSI_14'] = ta.rsi(df['Close_Price'], length=14)
        df['Historical_Volatility_20D'] = df['Close_Price'].pct_change().rolling(window=20).std() * (252**0.5)

        # Resample
        df_res = pd.DataFrame()
        df_res['EURUSD=X'] = df['Close_Price'].resample(FREQ).last()
        
        skip_cols = ['Close_Price']
        for col in df.columns:
            if col not in skip_cols:
                df_res[col] = df[col].resample(FREQ).last()
        
        df_res = df_res.ffill().dropna()

        # Lags
        numeric_cols = df_res.select_dtypes(include=np.number).columns
        for col in numeric_cols:
            df_res[f'{col}_Lag1'] = df_res[col].shift(1)
            df_res[f'{col}_Lag4'] = df_res[col].shift(4)

        # Last State
        last_actual_price = df_res['EURUSD=X'].iloc[-1]
        last_actual_date = df_res.index[-1]

        # Differencing
        cols_to_diff = [c for c in df_res.columns if 'Lag' not in c] 
        for col in cols_to_diff:
            df_res[f'{col}_D1'] = df_res[col].diff()

        df_res = df_res.dropna()
        return df_res, last_actual_price, last_actual_date

    def get_prediction_data(self):
        """
        Orchestrates fetching and predicting. 
        Returns dictionary with history and prediction.
        """
        try:
            raw_df = self.fetch_recent_data()
            processed_df, last_price, last_date = self.preprocess_for_inference(raw_df)
            
            predicted_price = last_price # Default to flat if no model
            
            if self.model_loaded:
                # Prepare Inference
                new_row = processed_df.iloc[[-1]].copy()
                new_row.index = [last_date + relativedelta(months=1)]
                inference_df = pd.concat([processed_df, new_row])
                inference_df['time_idx'] = np.arange(len(inference_df))
                inference_df['series'] = 'EURUSD'
                inference_df = inference_df.reset_index().rename(columns={'index': 'date'})

                pred_ds = TimeSeriesDataSet.from_parameters(
                    self.model.dataset_parameters, 
                    inference_df, 
                    predict=True, 
                    stop_randomization=True
                )
                pred_loader = pred_ds.to_dataloader(batch_size=1, shuffle=False)
                raw_prediction = self.model.predict(pred_loader, mode="prediction", return_x=False)
                predicted_change = raw_prediction[0][0].item()
                predicted_price = last_price + predicted_change
            else:
                # Demo Logic if model missing (for UI testing)
                import random
                change = random.uniform(-0.02, 0.02)
                predicted_price = last_price * (1 + change)

            target_date = (last_date + relativedelta(months=1)).date()

            # Prepare historical data for plotting (last 24 months)
            history = raw_df[['Close_Price']].iloc[-365:].copy() # Last year daily
            
            return {
                "history": history,
                "current_price": last_price,
                "predicted_price": predicted_price,
                "target_date": target_date,
                "success": True
            }
        except Exception as e:
            import traceback
            traceback.print_exc()
            return {"success": False, "error": str(e)}

# Initialize Forecaster
forecaster = ForexForecaster(MODEL_PATH)

# --- 3. DASHBOARD STYLING ---
# Professional Dark Theme Colors
COLORS = {
    'background': '#101115',
    'card_bg': '#1E2130',
    'text': '#E0E0E0',
    'accent': '#3498DB',
    'success': '#00C851',
    'danger': '#ff4444',
    'grid': '#2C3E50'
}

app = dash.Dash(__name__)
app.title = "Forex AI | Deep Learning Forecast"

# --- 4. LAYOUT ---
app.layout = html.Div(style={'backgroundColor': COLORS['background'], 'minHeight': '100vh', 'fontFamily': 'Segoe UI, sans-serif', 'color': COLORS['text'], 'padding': '20px'}, children=[
    
    # Header
    html.Div([
        html.Div([
            html.H1("Forex AI Forecast", style={'fontWeight': 'bold', 'margin': 0}),
            html.P("Temporal Fusion Transformer (TFT) Model", style={'opacity': 0.7, 'margin': 0})
        ]),
        html.Div([
            html.Span("Last Updated: ", style={'opacity': 0.7}),
            html.Span(datetime.now().strftime("%Y-%m-%d %H:%M"), style={'fontWeight': 'bold'})
        ], style={'textAlign': 'right'})
    ], style={'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center', 'marginBottom': '30px', 'borderBottom': f"1px solid {COLORS['grid']}", 'paddingBottom': '20px'}),

    # Controls Area
    html.Div([
        html.Label("Select Currency Perspective:", style={'fontWeight': 'bold', 'marginBottom': '10px', 'display': 'block'}),
        dcc.Dropdown(
            id='currency-pair-selector',
            options=[
                {'label': 'EUR to USD (Base Prediction)', 'value': 'EURUSD'},
                {'label': 'USD to EUR (Inverted)', 'value': 'USDEUR'}
            ],
            value='EURUSD',
            clearable=False,
            style={'color': '#000', 'width': '300px', 'marginBottom': '20px'}
        ),
        html.Button(
            "Run Live Prediction",
            id="predict-btn",
            n_clicks=0,
            style={
                'backgroundColor': COLORS['accent'],
                'color': 'white',
                'border': 'none',
                'padding': '12px 24px',
                'borderRadius': '5px',
                'fontSize': '16px',
                'cursor': 'pointer',
                'fontWeight': 'bold',
                'transition': '0.3s'
            }
        ),
    ], style={'backgroundColor': COLORS['card_bg'], 'padding': '20px', 'borderRadius': '10px', 'marginBottom': '20px'}),

    # Loading Wrapper
    dcc.Loading(
        id="loading-1",
        type="circle",
        color=COLORS['accent'],
        children=[
            html.Div(id='dashboard-content', style={'display': 'none'}, children=[
                
                # Metrics Row
                html.Div([
                    # Current Price Card
                    html.Div([
                        html.H4("Current Rate", style={'opacity': 0.7, 'margin': 0}),
                        html.H2(id='metric-current', style={'margin': '10px 0', 'fontWeight': 'bold'}),
                        html.P(id='metric-date', style={'fontSize': '12px', 'opacity': 0.5})
                    ], style={'flex': 1, 'backgroundColor': COLORS['card_bg'], 'padding': '20px', 'borderRadius': '10px', 'marginRight': '20px'}),
                    
                    # Predicted Price Card
                    html.Div([
                        html.H4("AI Prediction (Next Month)", style={'opacity': 0.7, 'margin': 0}),
                        html.H2(id='metric-predicted', style={'margin': '10px 0', 'fontWeight': 'bold', 'color': COLORS['accent']}),
                        html.P(id='metric-target-date', style={'fontSize': '12px', 'opacity': 0.5})
                    ], style={'flex': 1, 'backgroundColor': COLORS['card_bg'], 'padding': '20px', 'borderRadius': '10px', 'marginRight': '20px'}),
                    
                    # Change Card
                    html.Div([
                        html.H4("Forecasted Movement", style={'opacity': 0.7, 'margin': 0}),
                        html.H2(id='metric-change', style={'margin': '10px 0', 'fontWeight': 'bold'}),
                        html.P("Implied monthly volatility", style={'fontSize': '12px', 'opacity': 0.5})
                    ], style={'flex': 1, 'backgroundColor': COLORS['card_bg'], 'padding': '20px', 'borderRadius': '10px'}),
                ], style={'display': 'flex', 'justifyContent': 'space-between', 'marginBottom': '20px'}),

                # Graph Row
                html.Div([
                    dcc.Graph(id='forecast-graph', config={'displayModeBar': False}, style={'height': '500px'})
                ], style={'backgroundColor': COLORS['card_bg'], 'padding': '10px', 'borderRadius': '10px'})
            ])
        ]
    )
])

# --- 5. CALLBACKS ---
@app.callback(
    [Output('dashboard-content', 'style'),
     Output('metric-current', 'children'),
     Output('metric-date', 'children'),
     Output('metric-predicted', 'children'),
     Output('metric-target-date', 'children'),
     Output('metric-change', 'children'),
     Output('metric-change', 'style'),
     Output('forecast-graph', 'figure')],
    [Input('predict-btn', 'n_clicks'),
     Input('currency-pair-selector', 'value')]
)
def update_dashboard(n_clicks, pair):
    # Only run if button clicked or initial load (n_clicks 0)
    # We trigger initial load automatically
    
    data = forecaster.get_prediction_data()
    
    if not data['success']:
        # Return empty/error state
        return {'display': 'none'}, "Error", "-", "Error", "-", "-", {}, {}

    # Extract base data (Always EURUSD base)
    history_df = data['history']
    current_val = data['current_price']
    pred_val = data['predicted_price']
    target_date = data['target_date']
    
    # Handle Inversion if USDEUR selected
    if pair == 'USDEUR':
        # Invert History (1 / Rate)
        history_df['Close_Price'] = 1 / history_df['Close_Price']
        current_val = 1 / current_val
        pred_val = 1 / pred_val
        symbol = "USD/EUR"
    else:
        symbol = "EUR/USD"

    # Calculate Change
    pct_change = ((pred_val - current_val) / current_val) * 100
    change_text = f"{pct_change:+.2f}%"
    
    # Color for change
    change_style = {
        'margin': '10px 0', 
        'fontWeight': 'bold', 
        'color': COLORS['success'] if pct_change >= 0 else COLORS['danger']
    }

    # --- PLOTTING ---
    fig = go.Figure()

    # 1. Historical Line
    fig.add_trace(go.Scatter(
        x=history_df.index,
        y=history_df['Close_Price'],
        mode='lines',
        name='Historical',
        line=dict(color='#888888', width=2),
        hovertemplate='%{y:.4f}<extra></extra>'
    ))

    # 2. Connection Line (Dotted)
    last_hist_date = history_df.index[-1]
    fig.add_trace(go.Scatter(
        x=[last_hist_date, pd.Timestamp(target_date)],
        y=[current_val, pred_val],
        mode='lines',
        name='Projection',
        line=dict(color=COLORS['accent'], width=2, dash='dot'),
        showlegend=False
    ))

    # 3. Prediction Marker
    fig.add_trace(go.Scatter(
        x=[pd.Timestamp(target_date)],
        y=[pred_val],
        mode='markers+text',
        name='Forecast',
        marker=dict(color=COLORS['accent'], size=12, symbol='diamond'),
        text=[f"{pred_val:.4f}"],
        textposition="top center",
        textfont=dict(color=COLORS['text'])
    ))

    # Layout Styling
    fig.update_layout(
        title=f"{symbol} Trend & Forecast",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color=COLORS['text']),
        xaxis=dict(showgrid=True, gridcolor=COLORS['grid']),
        yaxis=dict(
            showgrid=True, 
            gridcolor=COLORS['grid'], 
            tickformat='.4f',
            zeroline=False
        ),
        margin=dict(l=40, r=40, t=40, b=40),
        hovermode="x unified"
    )

    return (
        {'display': 'block'}, # Show content
        f"{current_val:.4f}",
        f"As of {history_df.index[-1].strftime('%Y-%m-%d')}",
        f"{pred_val:.4f}",
        f"Target: {target_date.strftime('%B %Y')}",
        change_text,
        change_style,
        fig
    )

if __name__ == '__main__':
    app.run(debug=True)

Loading model from tft_model.ckpt...


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
